In [0]:
"""
Importa dependências
"""

import pyspark.sql.functions as F

In [0]:
"""
Carrega base de dados
"""

df = spark.read.format('delta').table('workspace.default.churn_modelling')

In [0]:
"""
Gera KPI's
"""

# KPI de faixa de idade ----------|
age_buckets = {
    '0. 18-': [0, 18],
    '1. 18-25': [18, 25],
    '2. 25-35': [25, 35],
    '3. 35-45': [35, 45],
    '4. 45-55': [45, 55],
    '5. 55-65': [55, 65],
    '6. 65+': [65, 1000]
}

expr = None
for label, bucket in age_buckets.items():
    if expr is None:
        expr = F.when(F.col('Age').between(bucket[0], bucket[1]), F.lit(label))
    else:
        expr = expr.when(F.col('Age').between(bucket[0], bucket[1]), F.lit(label))

df = df.withColumn('age_bucket', expr)


# KPI de faixa salarial ----------|
salary_buckets = {
    '1. Até 50k':      [0,      50000],
    '2. 50k-100k':     [50000,  100000],
    '3. 100k-150k':    [100000, 150000],
    '4. 150k-200k':    [150000, 200000],
    '5. Acima 200k':   [200000, 99999999]
}

expr = None
for label, bucket in salary_buckets.items():
    if expr is None:
        expr = F.when(F.col('EstimatedSalary').between(bucket[0], bucket[1]), F.lit(label))
    else:
        expr = expr.when(F.col('EstimatedSalary').between(bucket[0], bucket[1]), F.lit(label))

df = df.withColumn('salary_bucket', expr)


# KPI de faixa de saldo ----------|
balance_buckets = {
    '1. Zero':         [0,       0],
    '2. Até 50k':      [1,       50000],
    '3. 50k-100k':     [50000,   100000],
    '4. 100k-150k':    [100000,  150000],
    '5. 150k-200k':    [150000,  200000],
    '6. Acima 200k':   [200000,  99999999]
}

expr = None
for label, bucket in balance_buckets.items():
    if expr is None:
        expr = F.when(F.col('Balance').between(bucket[0], bucket[1]), F.lit(label))
    else:
        expr = expr.when(F.col('Balance').between(bucket[0], bucket[1]), F.lit(label))

df = df.withColumn('balance_bucket', expr)


# KPI de faixa de credit score ----|
credit_buckets = {
    '1. Muito Ruim  (<500)':  [0,   499],
    '2. Ruim        (500-579)':[500, 579],
    '3. Regular     (580-669)':[580, 669],
    '4. Bom         (670-739)':[670, 739],
    '5. Muito Bom   (740-799)':[740, 799],
    '6. Excelente   (800+)':  [800, 9999]
}

expr = None
for label, bucket in credit_buckets.items():
    if expr is None:
        expr = F.when(F.col('CreditScore').between(bucket[0], bucket[1]), F.lit(label))
    else:
        expr = expr.when(F.col('CreditScore').between(bucket[0], bucket[1]), F.lit(label))

df = df.withColumn('credit_score_bucket', expr)


# KPI de faixa de tenure ----------|
tenure_buckets = {
    '1. Novo        (0-1 anos)':  [0,  1],
    '2. Recente     (2-3 anos)':  [2,  3],
    '3. Intermediário (4-6 anos)':[4,  6],
    '4. Fiel        (7-9 anos)':  [7,  9],
    '5. Veterano    (10+ anos)':  [10, 9999]
}

expr = None
for label, bucket in tenure_buckets.items():
    if expr is None:
        expr = F.when(F.col('Tenure').between(bucket[0], bucket[1]), F.lit(label))
    else:
        expr = expr.when(F.col('Tenure').between(bucket[0], bucket[1]), F.lit(label))

df = df.withColumn('tenure_bucket', expr)

# KPI de razão saldo/salário --------|
df = df \
    .withColumn(
        'balance_salary_ratio',
        F.round(
            F.col('Balance') / F.when(F.col('EstimatedSalary') == 0, F.lit(1)).otherwise(F.col('EstimatedSalary')), 4
            )
        )

In [0]:
"""
Salva base
"""

df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.default.churn_modelling')